# Sync Odoo and Bexio Contacts

Bidirectional sync of contacts between Odoo (via JSON-2 API) and Bexio (via REST API).
On first run a full sync is performed; subsequent runs use delta sync based on `updated_at` / `write_date`.

## Setup

Load credentials from `.env`, create SSL context, and define helpers for Odoo JSON-2 and Bexio REST calls.

In [13]:
import json
import ssl
import urllib.request
import urllib.error
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------------
# Load credentials from .env
# ------------------------------------------------------------------
env_path = Path('.env')
ODOO_API_TOKEN = ''
BEXIO_API_TOKEN = ''
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line.startswith('ODOO_API_TOKEN='):
            ODOO_API_TOKEN = line.split('=', 1)[1]
        elif line.startswith('BEXIO_API_TOKEN='):
            BEXIO_API_TOKEN = line.split('=', 1)[1]

ODOO_URL = 'https://jupyter-lab.19.odoo.build'
ODOO_DB  = 'jupyter-lab'
BEXIO_API_URL = 'https://api.bexio.com/2.0'

ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

# File to remember last-sync timestamps for delta sync
LAST_SYNC_FILE = Path('tmp/last_sync.json')
LAST_SYNC_FILE.parent.mkdir(parents=True, exist_ok=True)

def load_last_sync():
    if LAST_SYNC_FILE.exists():
        return json.loads(LAST_SYNC_FILE.read_text())
    return {}

def save_last_sync(timestamps):
    LAST_SYNC_FILE.write_text(json.dumps(timestamps, indent=2, default=str))

last_sync = load_last_sync()
print('Setup complete. Last sync timestamps:', last_sync)


Setup complete. Last sync timestamps: {'bexio': '2026-09-01 15:52:37', 'odoo': '2026-09-01 13:52:43'}


## API Helpers

In [14]:
# ------------------------------------------------------------------
# Odoo JSON-2 helper
# ------------------------------------------------------------------
def odoo_json2(model, method, params):
    url = f"{ODOO_URL.rstrip('/')}/json/2/{model}/{method}"
    payload = json.dumps(params).encode()
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {ODOO_API_TOKEN}',
    }
    req = urllib.request.Request(url, data=payload, headers=headers)
    try:
        with urllib.request.urlopen(req, context=ssl_ctx) as resp:
            return json.loads(resp.read())
    except urllib.error.HTTPError as e:
        raise Exception(f'Odoo HTTP {e.code}: {e.read().decode()}')

# ------------------------------------------------------------------
# Bexio REST helper
# ------------------------------------------------------------------
def bexio_request(endpoint, method='GET', data=None):
    url = f"{BEXIO_API_URL.rstrip('/')}/{endpoint.lstrip('/')}"
    headers = {
        'Accept': 'application/json',
        'Authorization': f'Bearer {BEXIO_API_TOKEN}',
    }
    if data is not None:
        headers['Content-Type'] = 'application/json'
        payload = json.dumps(data).encode()
    else:
        payload = None
    req = urllib.request.Request(url, data=payload, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=ssl_ctx) as resp:
            return json.loads(resp.read())
    except urllib.error.HTTPError as e:
        raise Exception(f'Bexio HTTP {e.code}: {e.read().decode()}')

print('Helpers defined.')


Helpers defined.


## Test Connections

In [15]:
# Odoo
odoo_count = odoo_json2('res.partner', 'search_count', {'domain': []})
print(f'Odoo partners: {odoo_count}')

# Bexio
bexio_contacts_raw = bexio_request('/contact')
print(f'Bexio contacts: {len(bexio_contacts_raw)}')


Odoo partners: 41
Bexio contacts: 40


## Field Mapping

Mapping between Bexio `contact` fields and Odoo `res.partner` fields.

| Bexio field | Odoo field | Notes |
|-------------|------------|-------|
| `id` | `bexio_id` (stored as XML-ID) | Unique Bexio identifier |
| `name_1` | `name` | Company name or first name |
| `name_2` | `name` | Appended for persons (`name_1 name_2`) |
| `mail` | `email` | Primary e-mail address |
| `phone_fixed` | `phone` | Landline number |
| `phone_mobile` | `phone` | Mobile number (Odoo has no `mobile` field in this build) |
| `street_name` | `street` | Writable street field in Bexio |
| `address` | `street` | Read-only computed address in Bexio |
| `postcode` | `zip` | ZIP / postal code |
| `city` | `city` | City name |
| `country_id` | `country_id` | Bexio country ID ↔ Odoo country many2one |
| `contact_type_id` | `company_type` / `is_company` | `1` = company, `2` = person |
| `updated_at` | `write_date` | Used for delta sync |
| `nr` | `ref` | Bexio contact number || `remarks` | `comment` | Remarks / internal notes |


## Fetch Odoo Partners

In [16]:
ODOO_FIELDS = [
    'id', 'name', 'email', 'phone',
    'street', 'zip', 'city', 'country_id',
    'company_type', 'is_company', 'write_date', 'ref',
    'comment'
]

odoo_ids = odoo_json2('res.partner', 'search', {'domain': []})
print(f'Fetched {len(odoo_ids)} Odoo partner(s).')

odoo_partners = []
BATCH = 200
for i in range(0, len(odoo_ids), BATCH):
    batch = odoo_ids[i:i + BATCH]
    odoo_partners.extend(odoo_json2('res.partner', 'read', {'ids': batch, 'fields': ODOO_FIELDS}))

# Build lookup indexes
odoo_by_email = {}
odoo_by_name = {}
for p in odoo_partners:
    email = (p.get('email') or '').strip().lower()
    if email:
        odoo_by_email[email] = p
    name = (p.get('name') or '').strip().lower()
    if name:
        odoo_by_name[name] = p

print(f'Indexed {len(odoo_by_email)} by e-mail, {len(odoo_by_name)} by name.')


Fetched 41 Odoo partner(s).
Indexed 37 by e-mail, 40 by name.


## Fetch Bexio Contacts

In [17]:
# Bexio does not expose a queryable updated_at filter on the standard list endpoint,
# so we fetch all contacts and filter locally.
bexio_raw = bexio_request('/contact')
print(f'Fetched {len(bexio_raw)} Bexio contact(s).')

if last_sync.get('bexio'):
    cutoff = last_sync['bexio']
    bexio_contacts = [c for c in bexio_raw if c.get('updated_at') and c['updated_at'] > cutoff]
    print(f"Delta sync: {len(bexio_contacts)} Bexio contact(s) updated after {cutoff}")
else:
    bexio_contacts = bexio_raw
    print('Full sync: using all Bexio contacts.')

# Build lookup indexes
bexio_by_email = {}
bexio_by_name = {}
for c in bexio_contacts:
    email = (c.get('mail') or '').strip().lower()
    if email:
        bexio_by_email[email] = c
    name = ' '.join(p for p in [c.get('name_1', ''), c.get('name_2', '')] if p).strip().lower()
    if name:
        bexio_by_name[name] = c

print(f'Indexed {len(bexio_by_email)} by e-mail, {len(bexio_by_name)} by name.')


Fetched 40 Bexio contact(s).
Delta sync: 0 Bexio contact(s) updated after 2026-09-01 15:52:37
Indexed 0 by e-mail, 0 by name.


## Cache Country Mapping

Bexio stores `country_id` as an integer; Odoo uses a many2one.
We build bidirectional lookups via ISO-3166 alpha2 codes.

In [18]:
# 1. Fetch Bexio countries
bexio_countries = bexio_request('/country')
print(f'Loaded {len(bexio_countries)} Bexio countries.')

# 2. Fetch Odoo countries
odoo_country_ids = odoo_json2('res.country', 'search', {'domain': []})
odoo_countries = odoo_json2('res.country', 'read', {'ids': odoo_country_ids, 'fields': ['id', 'code']})
odoo_country_by_code = {c['code'].upper(): c['id'] for c in odoo_countries if c.get('code')}
print(f'Loaded {len(odoo_countries)} Odoo countries.')

# 3. Build Bexio country_id → Odoo country_id map
country_map = {}
reverse_country_map = {}
for bc in bexio_countries:
    code = (bc.get('iso_3166_alpha2') or '').upper()
    if code and code in odoo_country_by_code:
        country_map[bc['id']] = odoo_country_by_code[code]
        reverse_country_map[odoo_country_by_code[code]] = bc['id']

print(f'Mapped {len(country_map)} countries.')


Loaded 37 Bexio countries.


Loaded 251 Odoo countries.
Mapped 37 countries.


## XML-ID Helper

Store the Bexio contact ID as an Odoo external identifier (`ir.model.data`).

In [19]:
def set_bexio_xml_id(partner_id, bexio_id):
    """Create or update the Bexio XML-ID for a partner."""
    existing = odoo_json2('ir.model.data', 'search', {
        'domain': [('model', '=', 'res.partner'), ('res_id', '=', partner_id)]
    })
    if existing:
        odoo_json2('ir.model.data', 'write', {'ids': existing, 'vals': {'name': f'bexio_contact_{bexio_id}'}})
    else:
        odoo_json2('ir.model.data', 'create', {'vals_list': [{
            'name': f'bexio_contact_{bexio_id}',
            'model': 'res.partner',
            'res_id': partner_id,
            'module': '__sync__'
        }]}
    )
    return f'__sync__.bexio_contact_{bexio_id}'

print('XML-ID helpers defined.')


XML-ID helpers defined.


## Sync Logic

1. **Bexio → Odoo** — create or update Odoo partners from Bexio contacts.
2. **Odoo → Bexio** — update Bexio contacts from Odoo partners.

Matching strategy: **e-mail address** preferred; falls back to **name**.

In [20]:
# ------------------------------------------------------------------
# Transform helpers
# ------------------------------------------------------------------
def bexio_to_odoo_vals(contact):
    """Convert a Bexio contact dict to Odoo res.partner write values."""
    vals = {}
    name_parts = [contact.get('name_1', ''), contact.get('name_2', '')]
    name = ' '.join(p for p in name_parts if p).strip()
    if name:
        vals['name'] = name
    email = contact.get('mail')
    if email:
        vals['email'] = email.strip()
    phone = contact.get('phone_fixed') or contact.get('phone_mobile')
    if phone:
        vals['phone'] = phone
    if contact.get('address'):
        vals['street'] = contact['address']
    elif contact.get('street_name'):
        vals['street'] = contact['street_name']
    if contact.get('postcode'):
        vals['zip'] = contact['postcode']
    if contact.get('city'):
        vals['city'] = contact['city']
    if contact.get('country_id') and contact['country_id'] in country_map:
        vals['country_id'] = country_map[contact['country_id']]
    ctype = contact.get('contact_type_id')
    if ctype == 1:
        vals['company_type'] = 'company'
        vals['is_company'] = True
    elif ctype == 2:
        vals['company_type'] = 'person'
        vals['is_company'] = False
    if contact.get('nr'):
        vals['ref'] = str(contact['nr'])
    if contact.get('remarks'):
        vals['comment'] = contact['remarks']
    return vals

def odoo_to_bexio_vals(partner, for_create=False):
    """Convert an Odoo partner dict to Bexio contact write values."""
    vals = {}
    if partner.get('name'):
        names = partner['name'].split(' ', 1)
        vals['name_1'] = names[0]
        if len(names) > 1:
            vals['name_2'] = names[1]
    if partner.get('email'):
        vals['mail'] = partner['email'].strip()
    if partner.get('phone'):
        vals['phone_fixed'] = partner['phone']
    if partner.get('street'):
        vals['street_name'] = partner['street']
    if partner.get('zip'):
        vals['postcode'] = partner['zip']
    if partner.get('city'):
        vals['city'] = partner['city']
    # Map Odoo country_id back to Bexio country_id
    country = partner.get('country_id')
    if isinstance(country, list) and len(country) == 2:
        country = country[0]
    if country and country in reverse_country_map:
        vals['country_id'] = reverse_country_map[country]
    if partner.get('company_type') == 'company':
        vals['contact_type_id'] = 1
    else:
        vals['contact_type_id'] = 2
    if partner.get('comment'):
        vals['remarks'] = partner['comment']
    if for_create:
        vals['user_id'] = 1
        vals['owner_id'] = 1
    return vals

print('Transform helpers defined.')


Transform helpers defined.


### Bexio → Odoo

In [21]:
created_odoo = 0
updated_odoo = 0
skipped_odoo = 0

for contact in bexio_contacts:
    vals = bexio_to_odoo_vals(contact)
    if not vals.get('name'):
        skipped_odoo += 1
        continue

    email = (contact.get('mail') or '').strip().lower()
    name = vals['name'].lower()

    # Try e-mail match first, then name match
    partner = odoo_by_email.get(email) if email else None
    if not partner:
        partner = odoo_by_name.get(name)

    if partner:
        odoo_json2('res.partner', 'write', {'ids': [partner['id']], 'vals': vals})
        updated_odoo += 1
        set_bexio_xml_id(partner['id'], contact['id'])
    else:
        new_ids = odoo_json2('res.partner', 'create', {'vals_list': [vals]})
        new_id = new_ids[0] if isinstance(new_ids, list) else new_ids
        created_odoo += 1
        set_bexio_xml_id(new_id, contact['id'])
        # Update local indexes so Odoo → Bexio can skip it
        vals['id'] = new_id
        if vals.get('email'):
            odoo_by_email[vals['email'].lower()] = vals
        odoo_by_name[name] = vals

print(f'Bexio → Odoo: {created_odoo} created, {updated_odoo} updated, {skipped_odoo} skipped.')


Bexio → Odoo: 0 created, 0 updated, 0 skipped.


### Odoo → Bexio

In [22]:
created_bexio = 0
updated_bexio = 0
skipped_bexio = 0

# Re-fetch Bexio contacts to include any created in previous steps
bexio_raw_fresh = bexio_request('/contact')
bexio_fresh_by_email = {}
bexio_fresh_by_name = {}
for c in bexio_raw_fresh:
    em = (c.get('mail') or '').strip().lower()
    if em:
        bexio_fresh_by_email[em] = c
    nm = ' '.join(p for p in [c.get('name_1', ''), c.get('name_2', '')] if p).strip().lower()
    if nm:
        bexio_fresh_by_name[nm] = c

for partner in odoo_partners:
    # Delta sync: only push Odoo partners that changed since last sync
    if last_sync.get('odoo') and partner.get('write_date', '') <= last_sync['odoo']:
        continue

    vals = odoo_to_bexio_vals(partner, for_create=False)
    if not vals.get('name_1'):
        skipped_bexio += 1
        continue

    email = (partner.get('email') or '').strip().lower()
    name = (partner.get('name') or '').strip().lower()

    contact = bexio_fresh_by_email.get(email) if email else None
    if not contact:
        contact = bexio_fresh_by_name.get(name)

    if contact:
        bexio_request(f"/contact/{contact['id']}", method='POST', data=vals)
        updated_bexio += 1
    else:
        vals = odoo_to_bexio_vals(partner, for_create=True)
        bexio_request('/contact', method='POST', data=vals)
        created_bexio += 1

print(f'Odoo → Bexio: {created_bexio} created, {updated_bexio} updated, {skipped_bexio} skipped.')


Odoo → Bexio: 0 created, 4 updated, 0 skipped.


## Persist Sync Timestamps

In [23]:
# Record max Bexio updated_at from after the sync
bexio_raw_final = bexio_request('/contact')
bexio_timestamps = [c.get('updated_at', '') for c in bexio_raw_final if c.get('updated_at')]
last_sync['bexio'] = max(bexio_timestamps) if bexio_timestamps else ''

# Record current UTC time for Odoo
last_sync['odoo'] = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

save_last_sync(last_sync)
print(f'Sync timestamps saved: {last_sync}')


Sync timestamps saved: {'bexio': '2026-09-01 15:57:27', 'odoo': '2026-09-01 13:57:27'}


## Sync Summary

In [24]:
print('=' * 50)
print('SYNC SUMMARY')
print('=' * 50)
print(f"{'Direction':<20} {'Created':>8} {'Updated':>8} {'Skipped':>8}")
print('-' * 50)
print(f"{'Bexio → Odoo':<20} {created_odoo:>8} {updated_odoo:>8} {skipped_odoo:>8}")
print(f"{'Odoo → Bexio':<20} {created_bexio:>8} {updated_bexio:>8} {skipped_bexio:>8}")
print('=' * 50)
print(f"Next run will use delta sync from {last_sync}")


SYNC SUMMARY
Direction             Created  Updated  Skipped
--------------------------------------------------
Bexio → Odoo                0        0        0
Odoo → Bexio                0        4        0
Next run will use delta sync from {'bexio': '2026-09-01 15:57:27', 'odoo': '2026-09-01 13:57:27'}
